# PyPSA-DE: wiederverwendbarer Testlauf

Alle Angaben, die sich pro Lauf ändern, stehen **nur** in der Zelle *Einstellungen*.
Die übrigen Zellen werden unverändert ausgeführt.

Ablauf: Einstellungen → Repo kopieren → Optimieren → Speichern.

### Einstellungen

In [ ]:
from pathlib import Path

# ===================== PRO LAUF ANPASSEN =====================

# Ergebnisordner von Snakemake: results/<RUN_PREFIX>/<SCENARIO>/...
# Wird automatisch in die lokale Kopie von config/config.de.yaml geschrieben.
RUN_PREFIX = "20261029_mod5"

# Szenario aus config/scenarios.manual.yaml (muss in run: name stehen)
SCENARIO = "SR27_S3"

# Zu optimierende Jahre (myopisch, daher aufsteigend und lückenlos)
YEARS = [2025, 2030, 2035, 2040, 2045, 2050]

# Name des Ergebnisordners im Volume
VERSION = "modified-5"

# Vorhandenen Ergebnisordner im Volume überschreiben?
OVERWRITE = False

# ===================== SELTEN ANPASSEN =======================

REPO_SRC = Path("/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test")
LOCAL_DIR = Path("/local_disk0/pypsa-de-test")
VOLUME_RESULTS = Path(
    "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results"
)

CLUSTERS = 27
CORES = 8
MEM_MB = 60000
SKIP_DIRS = [".pixi", "scripts_LuKa"]

# ===================== ABGELEITET (nicht ändern) =============

NETWORK_DIR = LOCAL_DIR / "results" / RUN_PREFIX / SCENARIO / "networks"
DESTINATION = VOLUME_RESULTS / VERSION


def network_file(year):
    return f"base_s_{CLUSTERS}__none_{year}.nc"


print(f"Lauf:        {RUN_PREFIX} / {SCENARIO}")
print(f"Jahre:       {YEARS}")
print(f"Netzwerke:   {NETWORK_DIR}")
print(f"Speichern:   {DESTINATION}")

### Repo kopieren und Config setzen

In [ ]:
import os
import re
import shutil

# 1. Repository kopieren
if not REPO_SRC.is_dir():
    raise FileNotFoundError(f"Quellordner nicht gefunden: {REPO_SRC}")


def walk_error(error):
    raise error


for directory, dirs, files in os.walk(REPO_SRC, onerror=walk_error):
    directory = Path(directory)
    relative = directory.relative_to(REPO_SRC)

    # Diese Verzeichnisse direkt im Repo überspringen.
    if directory == REPO_SRC:
        for name in SKIP_DIRS:
            if name in dirs:
                print(f"Übersprungen: {name}", flush=True)
                dirs.remove(name)

    target = LOCAL_DIR / relative
    target.mkdir(parents=True, exist_ok=True)

    for name in files:
        # Vorhandene Zieldateien ersetzen.
        shutil.copyfile(directory / name, target / name)

print(f"Repo nach {LOCAL_DIR} kopiert.", flush=True)
print(f"pixi.toml vorhanden: {(LOCAL_DIR / 'pixi.toml').exists()}", flush=True)

# 2. RUN_PREFIX in die lokale Config schreiben (Repo bleibt unverändert)
config_file = LOCAL_DIR / "config" / "config.de.yaml"
text = config_file.read_text()

run_block = re.search(r"^run:\n(?:[ \t].*\n|\n)*", text, flags=re.M)
if run_block is None:
    raise ValueError("Block 'run:' in config.de.yaml nicht gefunden.")

block = run_block.group(0)
new_block, count = re.subn(
    r"^(\s+prefix:\s*).*$",
    lambda m: f'{m.group(1)}"{RUN_PREFIX}"',
    block,
    count=1,
    flags=re.M,
)
if count != 1:
    raise ValueError("Eintrag 'prefix:' im Block 'run:' nicht gefunden.")

# Szenario muss als aktiver Listeneintrag unter run: name stehen.
if not re.search(rf"^\s+-\s+{re.escape(SCENARIO)}\s*$", block, flags=re.M):
    raise ValueError(f"Szenario {SCENARIO} ist in run: name nicht aktiv.")

config_file.write_text(text.replace(block, new_block, 1))
print(f"run.prefix in lokaler Config gesetzt: {RUN_PREFIX}", flush=True)

### Optimieren

In [ ]:
import subprocess

# Installation, Umgebungsprüfung und Optimierung.
# Die Einstellungen werden als Umgebungsvariablen übergeben.
script = r"""
set -euxo pipefail

## Pixi installieren, falls noch nicht vorhanden
export PATH="$HOME/.pixi/bin:$PATH"

if ! command -v pixi >/dev/null 2>&1; then
  curl -fsSL https://pixi.sh/install.sh | sh
fi

pixi --version

## Lokales Arbeitsverzeichnis
cd "$LOCAL_DIR"

## Projektumgebung installieren
pixi install

## Python und Paketversionen prüfen
pixi run python -c '
import sys
import numpy
import geopandas

print("Python:", sys.executable)
print("NumPy:", numpy.__version__)
print("GeoPandas:", geopandas.__version__)
'

## Snakemake-Version prüfen
pixi run snakemake --version

## Szenariokonfigurationen erzeugen
pixi run snakemake -c1 build_scenarios

## Alle Jahre nacheinander optimieren
for YEAR in $YEARS; do

  TARGET="results/${RUN_PREFIX}/${SCENARIO}/networks/base_s_${CLUSTERS}__none_${YEAR}.nc"

  echo "=========================================="
  echo "Optimierung ${YEAR}"
  echo "Target: ${TARGET}"
  echo "=========================================="

  PYTHONPATH="" pixi run snakemake -c"$CORES" --jobs 1 \
    "$TARGET" \
    --resources mem_mb="$MEM_MB" \
    --set-resources solve_sector_network_myopic:mem_mb="$MEM_MB"

  echo "=== ${YEAR} erfolgreich abgeschlossen ==="

done
"""

env = dict(
    os.environ,
    LOCAL_DIR=str(LOCAL_DIR),
    RUN_PREFIX=RUN_PREFIX,
    SCENARIO=SCENARIO,
    YEARS=" ".join(str(year) for year in YEARS),
    CLUSTERS=str(CLUSTERS),
    CORES=str(CORES),
    MEM_MB=str(MEM_MB),
)

# Bash-Skript genau einmal ausführen.
# Ausgabe und Fehlermeldungen fortlaufend gemeinsam anzeigen.
with subprocess.Popen(
    ["bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    exit_code = process.wait()

# Bei einem Fehler auch die Notebook-Zelle fehlschlagen lassen.
if exit_code != 0:
    raise RuntimeError(
        f"Installation oder Modelllauf fehlgeschlagen: Exit-Code {exit_code}"
    )

print("Alle Schritte erfolgreich abgeschlossen.", flush=True)

### Speichern

In [ ]:
# Ergebnisse nach erfolgreicher Optimierung ins Volume kopieren

# Vor dem Kopieren prüfen, ob alle erwarteten Netzwerke vorliegen.
missing = [
    year for year in YEARS
    if not (NETWORK_DIR / network_file(year)).is_file()
]
if missing:
    raise FileNotFoundError(f"Ergebnisdateien fehlen für: {missing}")

# Bestehende Ergebnisse nicht versehentlich überschreiben.
if DESTINATION.exists() and any(DESTINATION.iterdir()) and not OVERWRITE:
    raise FileExistsError(
        f"{DESTINATION} enthält bereits Dateien. "
        "VERSION ändern oder OVERWRITE = True setzen."
    )

# Alle Dateien einschließlich Unterordner kopieren.
files = sorted(p for p in NETWORK_DIR.rglob("*") if p.is_file())

for file in files:
    target = DESTINATION / file.relative_to(NETWORK_DIR)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(file, target)

    # Einfache Kontrolle: Dateigrößen müssen übereinstimmen.
    if file.stat().st_size != target.stat().st_size:
        raise IOError(f"Dateigröße stimmt nicht überein: {target}")

    print(f"Gespeichert: {target}", flush=True)

print(f"{len(files)} Dateien nach {DESTINATION} kopiert.", flush=True)